# Transliteration with `indic-transliterate`

Convert text between a language's native script and roman (Latin) script while preserving pronunciation. Translation changes the *language*; transliteration only changes the *script*.

| | |
|---|---|
| Endpoint | `POST /transliterate` (JSON) |
| Input | `text`, `language` (22 Indic languages), optional `script` (`roman` default, or `native`), optional `max_output_tokens` |
| Output | `transliteration`, `language`, `script`, `truncated` |

Typical uses: rendering a Hindi SMS in Latin letters, turning "namaste" typed on an English keyboard into नमस्ते, normalising search queries.

In [ ]:
%pip install -q requests==2.32.3

In [ ]:
import os, json, requests

BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
API_KEY = os.environ["BODHAN_API_KEY"]  # export BODHAN_API_KEY=... before starting Jupyter
HEADERS = {"Authorization": f"Bearer {API_KEY}"}


def raise_for_bodhan(resp):
    """Bodhan errors are JSON: {"error": {"message", "code", "request_id"}}. Surface them readably."""
    if resp.ok:
        return resp
    try:
        err = resp.json()["error"]
        raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')} (request_id={err.get('request_id')})")
    except (ValueError, KeyError):
        resp.raise_for_status()

## 1. Native script to roman

In [ ]:
def transliterate(text: str, language: str, script: str = "roman") -> str:
    resp = requests.post(
        f"{BASE_URL}/transliterate",
        headers={**HEADERS, "Content-Type": "application/json"},
        json={"text": text, "language": language, "script": script},
        timeout=60,
    )
    return raise_for_bodhan(resp).json()["transliteration"]


transliterate("पनडुब्बी के अंदर पहुँच कर उसने राहत की साँस ली।", "hi")

## 2. Roman to native script

The direction people most often want in an input box: the user types in Latin letters, the app shows the native script.

In [ ]:
for lang, typed in [("hi", "aap kaise hain"), ("ta", "vanakkam nanba"), ("bn", "ami tomake bhalobashi")]:
    print(f"{lang}: {typed!r} -> {transliterate(typed, lang, script='native')}")

## 3. Languages

`as bn brx doi gu hi kn kok ks ks-Deva mai ml mni mni-Beng mr ne or pa sa sat sd sd-Arab ta te ur`

Note `language` is the language of the text, in both directions. `script` names the script you want *out*.

## 4. Transliteration vs translation

Given the English sentence "Good morning":

In [ ]:
translate_resp = requests.post(f"{BASE_URL}/translate", headers={**HEADERS, "Content-Type": "application/json"},
                               json={"text": "Good morning", "target_language": "hi"}, timeout=60)
hindi = raise_for_bodhan(translate_resp).json()["translation"]
print("translated   :", hindi)                            # सुप्रभात — a different language
print("transliterated:", transliterate(hindi, "hi"))      # suprabhaat — same language, roman letters

**Next:** combine with [`translate`](../translate/translate.ipynb) to produce roman-script output directly (`script="roman"` on `/translate`), which is one call instead of two.